# CerberusVision Phase 5 - Qwen-2.5-7B-Instruct (From Scratch QLoRA)

Bu defter, Phase 4.1'de yasadigimiz repetition loop (asiri tekrar) sorununu cozmek icin hazirlanmistir.
Model, augmented edilmis devasa (674 kayit) havuzla bastan (From Scratch) egitilmektedir.

**Kritik Ozellikler:**
- `DataCollatorForCompletionOnlyLM` ile EOS token garantisi.
- Pad Token catismasini onleme.
- Cosine LR Scheduler ile %10 Warmup (Buyuyen veri seti icin optimizasyon).
- Dry Run (Sanity Check) hucresi.

In [ ]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate
!pip install -q trl==0.9.6

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from transformers import TrainingArguments

model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_id)
# Ensure PAD token is not EOS to avoid masking clashes
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '<|endoftext|>'})

print("PAD Token ID:", tokenizer.pad_token_id)
print("EOS Token ID:", tokenizer.eos_token_id)
assert tokenizer.pad_token_id != tokenizer.eos_token_id, "PAD and EOS token ID cannot be the same!"
assert tokenizer.eos_token == "<|im_end|>", "EOS Token must be <|im_end|> for ChatML!"

In [ ]:
dataset = load_dataset("json", data_files="phase5_train.jsonl", split="train")

def format_chatml(example):
    messages = [
        {"role": "system", "content": "Extract shipping instruction data from OCR text as JSON."},
        {"role": "user", "content": str(example['input'])},
        {"role": "assistant", "content": str(example['output'])}
    ]
    # apply_chat_template handles adding <|im_start|> and <|im_end|> automatically
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chatml)

In [ ]:
# DRY RUN - SANITY CHECK
print("=== DRY RUN: Verifying EOS Token Masking ===")
response_template = "<|im_start|>assistant\n"
collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

sample = dataset[0]
tokenized = tokenizer(sample["text"], truncation=True, max_length=2048)
# Manually call the collator on a single batch to check labels
batch = collator([tokenized])
labels = batch["labels"][0].tolist()
input_ids = batch["input_ids"][0].tolist()

# Find where assistant starts
assistant_start = -1
for i, label in enumerate(labels):
    if label != -100:
        assistant_start = i
        break

print(f"Assistant response starts at index: {assistant_start}")
print("Last 10 Input Tokens:", tokenizer.convert_ids_to_tokens(input_ids[-10:]))
print("Last 10 Label IDs:", labels[-10:])

assert labels[-1] != -100, "FATAL: The EOS token is masked (-100) in the labels! Model will catastrophic forget!"
print("SUCCESS: EOS Token is properly labeled for loss calculation.")

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

training_args = TrainingArguments(
    output_dir="./qwen-2.5-7b-phase5-lora",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    logging_steps=10,
    max_steps=500, # Varies by dataset size, approx 1 epoch for 674 records
    warmup_ratio=0.1, # 10% Warmup for large augmented dataset
    lr_scheduler_type="cosine",
    save_steps=100,
    fp16=True,
    optim="paged_adamw_32bit"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
    data_collator=collator,
)

trainer.train()
trainer.model.save_pretrained("adapter_best")
tokenizer.save_pretrained("adapter_best")